In [1]:
# =============================================================================
# Kaggle Playground Series S6E4 — Predicting Irrigation Need
# Full EDA + LightGBM Baseline Pipeline
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score
)
from sklearn.inspection import permutation_importance
import lightgbm as lgb

In [3]:
# ── paths ──────────────────────────────────────────────────────────────────
TRAIN_PATH = "../dataset/train.csv"
TEST_PATH  = "../dataset/test.csv"
SUB_PATH   = "../dataset/sample_submission.csv"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
PALETTE = {"Low": "#1D9E75", "Medium": "#EF9F27", "High": "#D85A30"}

# =============================================================================
# 1. LOAD DATA
# =============================================================================

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
sub   = pd.read_csv(SUB_PATH)

print("=" * 60)
print(f"Train shape : {train.shape}")
print(f"Test shape  : {test.shape}")
print("=" * 60)

Train shape : (630000, 21)
Test shape  : (270000, 20)


In [4]:
# =============================================================================
# 2. BASIC EDA
# =============================================================================

print("\n── dtypes & nulls ──")
info = pd.DataFrame({
    "dtype":   train.dtypes,
    "nulls":   train.isnull().sum(),
    "pct_null": (train.isnull().mean() * 100).round(2),
    "n_unique": train.nunique()
})
print(info)

print("\n── target distribution ──")
print(train["Irrigation_Need"].value_counts(normalize=True).mul(100).round(1))

# ── target bar ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 3.5))
order = ["Low", "Medium", "High"]
counts = train["Irrigation_Need"].value_counts()[order]
bars = ax.bar(order, counts, color=[PALETTE[o] for o in order], width=0.5, edgecolor="white")
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 80,
            f"{count:,}", ha="center", va="bottom", fontsize=10)
ax.set_title("Target class distribution", fontsize=13, fontweight="medium")
ax.set_ylabel("Count")
ax.set_xlabel("")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "01_target_dist.png", dpi=150)
plt.close()

# ── numeric distributions ────────────────────────────────────────────────────
NUM_COLS = [
    "Soil_pH", "Soil_Moisture", "Organic_Carbon", "Electrical_Conductivity",
    "Temperature_C", "Humidity", "Rainfall_mm", "Sunlight_Hours",
    "Wind_Speed_kmh", "Field_Area_hectare", "Previous_Irrigation_mm"
]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()
for i, col in enumerate(NUM_COLS):
    for label, grp in train.groupby("Irrigation_Need"):
        axes[i].hist(grp[col], bins=40, alpha=0.55,
                     color=PALETTE[label], label=label, density=True)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_yticks([])
axes[-1].axis("off")
handles = [plt.Rectangle((0,0),1,1, color=PALETTE[l]) for l in order]
fig.legend(handles, order, loc="lower right", ncol=3, frameon=False, fontsize=10)
fig.suptitle("Numeric feature distributions by Irrigation_Need",
             fontsize=13, fontweight="medium", y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "02_numeric_dists.png", dpi=150, bbox_inches="tight")
plt.close()

# ── correlation heatmap ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
corr = train[NUM_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, linewidths=0.4, ax=ax, annot_kws={"size": 8})
ax.set_title("Numeric feature correlation matrix", fontsize=13, fontweight="medium")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03_correlation.png", dpi=150)
plt.close()

# ── categorical analysis ──────────────────────────────────────────────────────
CAT_COLS = [
    "Soil_Type", "Crop_Type", "Crop_Growth_Stage", "Season",
    "Irrigation_Type", "Water_Source", "Region", "Mulching_Used"
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(CAT_COLS):
    ct = (train.groupby([col, "Irrigation_Need"])
               .size()
               .unstack(fill_value=0)
               .reindex(columns=order))
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct.plot(kind="bar", stacked=True, ax=axes[i],
                color=[PALETTE[o] for o in order], edgecolor="white", linewidth=0.3)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel("")
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=30, ha="right", fontsize=8)
    axes[i].legend_.remove()
    axes[i].set_ylabel("% share")
handles = [plt.Rectangle((0,0),1,1, color=PALETTE[l]) for l in order]
fig.legend(handles, order, loc="upper right", ncol=3, frameon=False, fontsize=10)
fig.suptitle("Target share within each categorical feature",
             fontsize=13, fontweight="medium")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "04_categorical_analysis.png", dpi=150, bbox_inches="tight")
plt.close()

# ── leakage check: Irrigation_Type ──────────────────────────────────────────
print("\n── Irrigation_Type vs Irrigation_Need (leakage check) ──")
leak_check = pd.crosstab(train["Irrigation_Type"], train["Irrigation_Need"],
                          normalize="index").mul(100).round(1)
print(leak_check)


── dtypes & nulls ──
                           dtype  nulls  pct_null  n_unique
id                         int64      0       0.0    630000
Soil_Type                 object      0       0.0         4
Soil_pH                  float64      0       0.0       341
Soil_Moisture            float64      0       0.0      5223
Organic_Carbon           float64      0       0.0       131
Electrical_Conductivity  float64      0       0.0       341
Temperature_C            float64      0       0.0      2934
Humidity                 float64      0       0.0      6475
Rainfall_mm              float64      0       0.0     19308
Sunlight_Hours           float64      0       0.0       701
Wind_Speed_kmh           float64      0       0.0      1935
Crop_Type                 object      0       0.0         6
Crop_Growth_Stage         object      0       0.0         4
Season                    object      0       0.0         3
Irrigation_Type           object      0       0.0         4
Water_Source      

In [5]:
# =============================================================================
# 3. FEATURE ENGINEERING
# =============================================================================

def engineer(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # ── ordinal encode growth stage ──────────────────────────────────────────
    stage_order = ["Sowing", "Vegetative", "Flowering", "Harvesting"]
    stage_map   = {s: i for i, s in enumerate(stage_order)}
    df["growth_stage_ord"] = df["Crop_Growth_Stage"].map(stage_map).fillna(2)

    # ── physics-motivated features ───────────────────────────────────────────
    # evapotranspiration proxy (Penman–Monteith simplified)
    df["et_proxy"]          = df["Temperature_C"] * df["Sunlight_Hours"] / (df["Humidity"] + 1)
    # water availability
    df["water_supply"]      = df["Rainfall_mm"] + df["Previous_Irrigation_mm"]
    df["moisture_deficit"]  = df["et_proxy"] - df["Soil_Moisture"]
    df["rain_per_moisture"] = df["Rainfall_mm"] / (df["Soil_Moisture"] + 1)
    # salinity × pH stress
    df["soil_stress"]       = df["Electrical_Conductivity"] * (df["Soil_pH"] - 7).abs()
    # wind amplifies drying
    df["drying_index"]      = df["Wind_Speed_kmh"] * df["Temperature_C"] / (df["Humidity"] + 1)
    # total area demand
    df["area_demand"]       = df["Field_Area_hectare"] * df["et_proxy"]
    # pH deviation from neutral
    df["ph_dev"]            = (df["Soil_pH"] - 7).abs()

    # ── interaction features ─────────────────────────────────────────────────
    df["temp_x_sun"]        = df["Temperature_C"] * df["Sunlight_Hours"]
    df["hum_x_rain"]        = df["Humidity"] * df["Rainfall_mm"]
    df["crop_stage"]        = df["Crop_Type"].astype(str) + "_" + df["Crop_Growth_Stage"].astype(str)
    df["region_season"]     = df["Region"].astype(str) + "_" + df["Season"].astype(str)

    return df


train = engineer(train)
test  = engineer(test)

In [6]:
# =============================================================================
# 4. ENCODING
# =============================================================================

TARGET = "Irrigation_Need"
ID_COL = "id"
DROP_COLS = [ID_COL, TARGET, "Crop_Growth_Stage"]  # ordinal encoded version kept

target_enc = LabelEncoder()
y = target_enc.fit_transform(train[TARGET])
print(f"\nTarget classes: {list(target_enc.classes_)} → {list(range(len(target_enc.classes_)))}")

# Features to use
CAT_HIGH_CARD = ["crop_stage", "region_season"]
CAT_LOW_CARD  = [c for c in train.select_dtypes("object").columns
                 if c not in [TARGET, "Crop_Growth_Stage"]]

# Label-encode all categoricals (LightGBM handles them natively)
all_cats = CAT_LOW_CARD + CAT_HIGH_CARD
le_dict  = {}
for col in all_cats:
    if col in train.columns:
        le = LabelEncoder()
        combined = pd.concat([train[col], test[col]], axis=0).astype(str)
        le.fit(combined)
        train[col] = le.transform(train[col].astype(str))
        test[col]  = le.transform(test[col].astype(str))
        le_dict[col] = le

feature_cols = [c for c in train.columns if c not in DROP_COLS and c != ID_COL]
X      = train[feature_cols]
X_test = test[feature_cols]

print(f"\nTotal features used: {len(feature_cols)}")
print(f"Engineered features added: {len(feature_cols) - 20}")


Target classes: ['High', 'Low', 'Medium'] → [0, 1, 2]

Total features used: 31
Engineered features added: 11


In [7]:
# =============================================================================
# 5. LIGHTGBM CROSS-VALIDATION
# =============================================================================

LGBM_PARAMS = {
    "objective":        "multiclass",
    "num_class":        3,
    "metric":           "multi_logloss",
    "learning_rate":    0.05,
    "num_leaves":       127,
    "max_depth":        -1,
    "min_child_samples": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq":     5,
    "reg_alpha":        0.1,
    "reg_lambda":       0.1,
    "n_estimators":     1000,
    "random_state":     42,
    "verbose":          -1,
    "n_jobs":           -1,
}

SKF  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds   = np.zeros((len(X), 3))
test_preds  = np.zeros((len(X_test), 3))
fold_scores = []

print("\n" + "=" * 60)
print("LightGBM 5-Fold Cross-Validation")
print("=" * 60)

for fold, (tr_idx, val_idx) in enumerate(SKF.split(X, y), 1):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y[tr_idx],      y[val_idx]

    model = lgb.LGBMClassifier(**LGBM_PARAMS)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(period=-1)
        ]
    )

    val_proba = model.predict_proba(X_val)
    val_pred  = np.argmax(val_proba, axis=1)
    score     = f1_score(y_val, val_pred, average="macro")

    oof_preds[val_idx]  = val_proba
    test_preds         += model.predict_proba(X_test) / SKF.n_splits
    fold_scores.append(score)

    print(f"  Fold {fold} | macro F1 = {score:.5f} | best iter = {model.best_iteration_}")

oof_labels = np.argmax(oof_preds, axis=1)
oof_f1     = f1_score(y, oof_labels, average="macro")
oof_acc    = accuracy_score(y, oof_labels)

print(f"\n{'─'*60}")
print(f"  OOF macro F1 : {oof_f1:.5f}")
print(f"  OOF accuracy : {oof_acc:.5f}")
print(f"  Fold F1s     : {[round(s,5) for s in fold_scores]}")
print(f"  Std dev      : {np.std(fold_scores):.5f}")
print("=" * 60)


LightGBM 5-Fold Cross-Validation
  Fold 1 | macro F1 = 0.96924 | best iter = 357
  Fold 2 | macro F1 = 0.96996 | best iter = 370
  Fold 3 | macro F1 = 0.96937 | best iter = 342
  Fold 4 | macro F1 = 0.96890 | best iter = 354
  Fold 5 | macro F1 = 0.96990 | best iter = 410

────────────────────────────────────────────────────────────
  OOF macro F1 : 0.96947
  OOF accuracy : 0.98442
  Fold F1s     : [0.96924, 0.96996, 0.96937, 0.9689, 0.9699]
  Std dev      : 0.00040


In [8]:
# =============================================================================
# 6. EVALUATION PLOTS
# =============================================================================

# ── confusion matrix ─────────────────────────────────────────────────────────
cm   = confusion_matrix(y, oof_labels)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="YlGn",
            xticklabels=target_enc.classes_,
            yticklabels=target_enc.classes_,
            linewidths=0.5, ax=ax)
ax.set_title(f"OOF Confusion Matrix  (macro F1 = {oof_f1:.4f})",
             fontsize=12, fontweight="medium")
ax.set_ylabel("True")
ax.set_xlabel("Predicted")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "05_confusion_matrix.png", dpi=150)
plt.close()

print("\n── Classification report ──")
print(classification_report(y, oof_labels, target_names=target_enc.classes_))

# ── feature importance ───────────────────────────────────────────────────────
imp_df = (pd.DataFrame({
    "feature":    feature_cols,
    "importance": model.feature_importances_
})
.sort_values("importance", ascending=False)
.head(25))

fig, ax = plt.subplots(figsize=(8, 9))
colors = ["#1D9E75" if "proxy" in f or "deficit" in f or "supply" in f
          or f in ["et_proxy","water_supply","moisture_deficit","rain_per_moisture",
                   "drying_index","area_demand","soil_stress","temp_x_sun","hum_x_rain"]
          else "#378ADD" for f in imp_df["feature"]]
ax.barh(imp_df["feature"][::-1], imp_df["importance"][::-1],
        color=colors[::-1], edgecolor="white")
ax.set_title("Top-25 feature importances (last fold)", fontsize=13, fontweight="medium")
ax.set_xlabel("LightGBM gain importance")
# legend
from matplotlib.patches import Patch
legend_els = [Patch(facecolor="#1D9E75", label="Engineered"),
              Patch(facecolor="#378ADD", label="Original")]
ax.legend(handles=legend_els, frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "06_feature_importance.png", dpi=150)
plt.close()


── Classification report ──
              precision    recall  f1-score   support

        High       0.97      0.91      0.94     21009
         Low       0.99      0.99      0.99    369917
      Medium       0.98      0.97      0.98    239074

    accuracy                           0.98    630000
   macro avg       0.98      0.96      0.97    630000
weighted avg       0.98      0.98      0.98    630000



In [12]:
# =============================================================================
# 7. SUBMISSION
# =============================================================================

test_labels = target_enc.inverse_transform(np.argmax(test_preds, axis=1))
sub[TARGET]  = test_labels
sub.to_csv("outputs/submission.csv", index=False)

print(f"\nSubmission saved → {OUTPUT_DIR}/submission.csv")
print(sub[TARGET].value_counts())


Submission saved → outputs/submission.csv
Irrigation_Need
Low       159805
Medium    101581
High        8614
Name: count, dtype: int64


In [11]:
sub.head()

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


In [ ]:
# =============================================================================
# 8. OPTUNA HYPERPARAMETER OPTIMIZATION
# =============================================================================

# import optuna
# optuna.logging.set_verbosity(optuna.logging.WARNING)
#
# def objective(trial):
#     params = {
#         "objective":         "multiclass",
#         "num_class":         3,
#         "metric":            "multi_logloss",
#         "learning_rate":     trial.suggest_float("lr",        0.01, 0.1,  log=True),
#         "num_leaves":        trial.suggest_int("num_leaves",  31,   255),
#         "min_child_samples": trial.suggest_int("min_child",   10,   100),
#         "feature_fraction":  trial.suggest_float("ff",        0.5,  1.0),
#         "bagging_fraction":  trial.suggest_float("bf",        0.5,  1.0),
#         "bagging_freq":      trial.suggest_int("bfreq",       1,    10),
#         "reg_alpha":         trial.suggest_float("ra",        1e-4, 10.0, log=True),
#         "reg_lambda":        trial.suggest_float("rl",        1e-4, 10.0, log=True),
#         "n_estimators":      1000,
#         "random_state":      42,
#         "verbose":           -1,
#         "n_jobs":            -1,
#     }
#     scores = []
#     for tr_idx, val_idx in SKF.split(X, y):
#         m = lgb.LGBMClassifier(**params)
#         m.fit(X.iloc[tr_idx], y[tr_idx],
#               eval_set=[(X.iloc[val_idx], y[val_idx])],
#               callbacks=[lgb.early_stopping(30, verbose=False),
#                          lgb.log_evaluation(-1)])
#         pred = np.argmax(m.predict_proba(X.iloc[val_idx]), axis=1)
#         scores.append(f1_score(y[val_idx], pred, average="macro"))
#     return np.mean(scores)
#
# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=50, show_progress_bar=True)
# print("Best params:", study.best_params)
# print("Best OOF F1:", study.best_value)